# Final M6 holdout run: Financial Chronos with selected sampling settings

Runs the sampling configuration chosen on the six pre-M6 validation origins over
the 12 official M6 rounds. This is the holdout evaluation, not another
validation sweep. The weights stay frozen and only the sampling settings differ
from the project's original Financial Chronos run.

| | |
|---|---|
| Model | `FinText/Chronos_Small_2021_Global`, zero-shot, frozen |
| Temperature | **5.0** (selected) |
| top-k | **full** = 4096, the full vocabulary (selected) |
| top-p | 1.0 |
| Fixed | context 512 weekdays, horizon 20, 500 samples, seed 42, official M6 asset order |
| Input | the 12 trusted M6 round contexts on Drive |
| Output | 12 NPZ files, one per round |

The configuration was selected by lowest mean RPS across the six validation
origins, whose targets all end on or before 2022-03-04, before the first scored
M6 target on 2022-03-07. No M6 outcome was used in the selection.

Only the selected configuration runs here. The default settings are not rerun:
the existing Financial Chronos 500-sample M6 result of 0.177947
already provides that comparison, provided its setup is confirmed equivalent.

This notebook saves raw forecasts only. It computes no RPS, ECE, confidence or
entropy, and applies no shrinkage or other calibration. Scoring happens later
with the project's existing local code.

Expect roughly 4 minutes per round, so about 45 minutes for all 12. Every round
is written and verified before the next begins, and re-running skips finished
rounds.

## 1. Install Chronos

In [ ]:
%pip install -q chronos-forecasting

## 2. Imports and settings

The sampling settings are the selected ones. Everything else, including the
context length, horizon, sample count, seed, batch size, chunking and asset
order, is carried over unchanged from the established Financial Chronos runs, so
the only difference against the original M6 result is the sampling.

In [ ]:
import json
import traceback
from datetime import datetime, timezone
from importlib.metadata import version
from inspect import signature
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from chronos import ChronosPipeline

# --- Model (frozen, inference only) ----------------------------------------
MODEL_ID = "FinText/Chronos_Small_2021_Global"

# --- Selected sampling settings, from the validation experiment -------------
TEMPERATURE = 5.0
TOP_K_SETTING = "full_vocabulary"
TOP_K_LABEL = "full"
# Resolved from the checkpoint at load time and checked against this value,
# which is the one the validation grid actually ran with.
EXPECTED_RESOLVED_TOP_K = 4096
TOP_P = 1.0

# --- Everything else is fixed, exactly as in the established runs -----------
NUM_SAMPLES = 500
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 20
RANDOM_SEED = 42
SERIES_BATCH_SIZE = 10                # lower ONLY on CUDA OOM

SAMPLE_CHUNK_SIZE = 100
N_CHUNKS, remainder = divmod(NUM_SAMPLES, SAMPLE_CHUNK_SIZE)
assert remainder == 0, "NUM_SAMPLES must be a whole number of chunks"

ROUNDS = range(1, 13)

# Official M6 round schedule, unchanged.
# round -> context start, origin (= context end), forecast start, forecast end
ROUND_SCHEDULE = {
    1:  ("2020-03-19", "2022-03-04", "2022-03-07", "2022-04-01"),
    2:  ("2020-04-16", "2022-04-01", "2022-04-04", "2022-04-29"),
    3:  ("2020-05-14", "2022-04-29", "2022-05-02", "2022-05-27"),
    4:  ("2020-06-11", "2022-05-27", "2022-05-30", "2022-06-24"),
    5:  ("2020-07-09", "2022-06-24", "2022-06-27", "2022-07-22"),
    6:  ("2020-08-06", "2022-07-22", "2022-07-25", "2022-08-19"),
    7:  ("2020-09-03", "2022-08-19", "2022-08-22", "2022-09-16"),
    8:  ("2020-10-01", "2022-09-16", "2022-09-19", "2022-10-14"),
    9:  ("2020-10-29", "2022-10-14", "2022-10-17", "2022-11-11"),
    10: ("2020-11-26", "2022-11-11", "2022-11-14", "2022-12-09"),
    11: ("2020-12-24", "2022-12-09", "2022-12-12", "2023-01-06"),
    12: ("2021-01-21", "2023-01-06", "2023-01-09", "2023-02-03"),
}

# Genuine leading missing history: CARR and OGN listed late, so their early
# returns do not exist. They stay NaN and are handled by the model's
# missing-value mask, never filled or dropped.
EXPECTED_LEADING_NAN = {r: {"OGN": 302 - 20 * (r - 1)} for r in ROUNDS}
EXPECTED_LEADING_NAN[1] = {"CARR": 1, "OGN": 302}

# Official M6 asset order - used to verify the contexts, not to reorder them.
OFFICIAL_ASSET_ORDER = [
    "ABBV", "ACN", "AEP", "AIZ", "ALLE", "AMAT", "AMP", "AMZN", "AVB", "AVY",
    "AXP", "BDX", "BF-B", "BMY", "BR", "CARR", "CDW", "CE", "CHTR", "CNC",
    "CNP", "COP", "CTAS", "CZR", "DG", "DPZ", "DRE", "DXC", "EWA", "EWC",
    "EWG", "EWH", "EWJ", "EWL", "EWQ", "EWT", "EWU", "EWY", "EWZ", "FTV",
    "GOOG", "GPC", "GSG", "HIG", "HIGH.L", "HST", "HYG", "IAU", "ICLN",
    "IEAA.L", "IEF", "IEFM.L", "IEMG", "IEUS", "IEVL.L", "IGF", "INDA",
    "IUMO.L", "IUVL.L", "IVV", "IWM", "IXN", "JPEA.L", "JPM", "KR", "LQD",
    "MCHI", "META", "MVEU.L", "OGN", "PG", "PPL", "PRU", "PYPL", "RE",
    "REET", "ROL", "ROST", "SEGA.L", "SHY", "SLV", "SPMV.L", "TLT", "UNH",
    "URI", "V", "VRSK", "VXX", "WRK", "XLB", "XLC", "XLE", "XLF", "XLI",
    "XLK", "XLP", "XLU", "XLV", "XLY", "XOM",
]
N_ASSETS = len(OFFICIAL_ASSET_ORDER)

_predict_params = signature(ChronosPipeline.predict).parameters
_missing = [p for p in ("temperature", "top_k", "top_p") if p not in _predict_params]
assert not _missing, (
    f"This chronos-forecasting build does not accept {_missing} in predict(); "
    "the selected sampling settings cannot be applied."
)

print(f"chronos-forecasting {version('chronos-forecasting')} | torch {torch.__version__}")
print(f"Final M6 holdout run: {MODEL_ID}")
print(f"Selected sampling: temperature {TEMPERATURE}, top_k {TOP_K_SETTING} "
      f"(expect {EXPECTED_RESOLVED_TOP_K}), top_p {TOP_P}")
print(f"Fixed: {NUM_SAMPLES} samples in {N_CHUNKS} chunks of {SAMPLE_CHUNK_SIZE}, "
      f"context {CONTEXT_LENGTH}, horizon {PREDICTION_LENGTH}, seed {RANDOM_SEED}, "
      f"{N_ASSETS} assets, rounds 1-12")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Configure paths

Contexts come from the same shared M6 folder every previous run used. Outputs go
to a folder named after the selected configuration, kept well away from the
original {INFO['display']} results so nothing can be overwritten or confused.

In [ ]:
# Existing shared M6 context folder - the same 12 files as every previous run.
DRIVE_CONTEXT_DIR = Path("/content/drive/MyDrive/HonoursResearch/Round_1_Context")

# >>> THE ONLY NEW PATH <<<  separate from the original Financial Chronos outputs
DRIVE_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/HonoursResearch/outputs/financial_chronos"
    "/temperature_m6/temp5.0_topkfull")

OUTPUT_PREFIX = "financial_chronos"


def context_path(round_number: int) -> Path:
    return DRIVE_CONTEXT_DIR / f"round_{round_number:02d}_context.csv"


def samples_path(round_number: int) -> Path:
    return DRIVE_OUTPUT_DIR / (
        f"{OUTPUT_PREFIX}_round{round_number:02d}"
        f"_temp{TEMPERATURE:.1f}_topk{TOP_K_LABEL}_samples{NUM_SAMPLES}.npz")


if not DRIVE_CONTEXT_DIR.is_dir():
    raise FileNotFoundError(
        f"Context folder not found:\n  {DRIVE_CONTEXT_DIR}\n"
        "It must be the SAME folder used by the main Financial Chronos notebook, "
        "containing round_01_context.csv .. round_12_context.csv."
    )

missing = [context_path(r).name for r in ROUNDS if not context_path(r).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing context files in {DRIVE_CONTEXT_DIR}:\n  " + "\n  ".join(missing))

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Contexts (existing, shared): {DRIVE_CONTEXT_DIR}")
print(f"Output folder (new)        : {DRIVE_OUTPUT_DIR}")
print(f"All {len(list(ROUNDS))} M6 context files present")
print(f"\nExample filename: {samples_path(1).name}")

## 5. GPU check

A GPU is required. This cell stops rather than silently running on CPU.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Runtime > Change runtime type > Hardware "
        "accelerator: GPU, then re-run from section 3."
    )

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
GPU_NAME = torch.cuda.get_device_name(0)

print(f"GPU   : {GPU_NAME}")
print(f"dtype : {DTYPE}")

## 6. Load the model, once, and confirm the resolved top-k

Same checkpoint, CUDA and dtype setup as the established runs. Loaded here and
reused for every round, never reloaded inside the loop.

The full-vocabulary top-k is read from the loaded checkpoint and checked against
the value the validation grid ran with. If they differ, this run would not be
using the configuration that was actually selected, so it stops.

In [ ]:
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

pipeline = ChronosPipeline.from_pretrained(
    MODEL_ID,
    device_map="cuda",
    torch_dtype=DTYPE,
)

n_params = sum(p.numel() for p in pipeline.model.parameters())
MODEL_LOADED_ONCE = True


def resolve_full_vocabulary_top_k(loaded) -> int:
    """This checkpoint's token-vocabulary size, for untruncated sampling."""
    chronos_config = getattr(loaded.model, "config", None)
    n_tokens = getattr(chronos_config, "n_tokens", None)
    if n_tokens is None:
        raw_config = getattr(getattr(loaded.model, "model", None), "config", None)
        raw_chronos = getattr(raw_config, "chronos_config", None)
        if isinstance(raw_chronos, dict):
            n_tokens = raw_chronos.get("n_tokens")
    if n_tokens is None:
        raise RuntimeError(
            "Could not read n_tokens from the loaded model. Inspect "
            "pipeline.model.config rather than guessing a value."
        )
    return int(n_tokens)


RESOLVED_TOP_K = (resolve_full_vocabulary_top_k(pipeline)
                  if TOP_K_SETTING == "full_vocabulary" else int(TOP_K_SETTING))

assert RESOLVED_TOP_K == EXPECTED_RESOLVED_TOP_K, (
    f"Resolved top_k {RESOLVED_TOP_K} does not match the validated "
    f"{EXPECTED_RESOLVED_TOP_K}. This run would not be the selected configuration."
)

print(f"Loaded: {MODEL_ID}")
print(f"  parameters   : {n_params:,} ({n_params / 1e6:.1f}M)")
print(f"  device       : {GPU_NAME} | dtype {DTYPE}")
print(f"  sampling     : temperature {TEMPERATURE}, top_k {RESOLVED_TOP_K}, top_p {TOP_P}")
print(f"  top_k check  : matches the validated {EXPECTED_RESOLVED_TOP_K}")
print("Model loaded ONCE - reused for all 12 rounds.")

## 7. Helpers

The same three steps as the established runs. `load_and_validate_context` checks
one M6 context's shape, dates, asset order and missing-value pattern.
`generate_round_samples` draws 500 trajectories in chunks with the seed set once
per round. `save_and_verify_round` writes the NPZ with its run metadata and
reads it back.

The first three array keys match the established format, so the project's
existing scoring code reads these files unchanged.

In [ ]:
def load_and_validate_context(round_number: int):
    """Load one M6 context, validate it, and build the model input."""
    ctx_start, origin, fc_start, fc_end = ROUND_SCHEDULE[round_number]
    df = pd.read_csv(context_path(round_number), parse_dates=["date"])

    assert df.shape[0] == CONTEXT_LENGTH, (
        f"Round {round_number}: expected {CONTEXT_LENGTH} rows, found {df.shape[0]}"
    )
    assert list(df.columns) == ["date"] + OFFICIAL_ASSET_ORDER, (
        f"Round {round_number}: columns are not 'date' + the official M6 asset order"
    )
    dates = df["date"]
    assert dates.is_monotonic_increasing, f"Round {round_number}: dates not ascending"
    assert not dates.duplicated().any(), f"Round {round_number}: duplicate dates"
    assert dates.iloc[0] == pd.Timestamp(ctx_start), f"Round {round_number}: context start"
    assert dates.iloc[-1] == pd.Timestamp(origin), f"Round {round_number}: context end/origin"
    assert (dates <= pd.Timestamp(origin)).all(), (
        f"Round {round_number}: context contains a date after the origin"
    )

    matrix = df[OFFICIAL_ASSET_ORDER].to_numpy(dtype=np.float64).T
    assert matrix.shape == (N_ASSETS, CONTEXT_LENGTH)
    assert np.array_equal(
        matrix, df[OFFICIAL_ASSET_ORDER].to_numpy().T, equal_nan=True
    ), f"Round {round_number}: model input differs from the loaded context values"

    leading_missing, interior_missing = {}, {}
    for i, symbol in enumerate(OFFICIAL_ASSET_ORDER):
        row = matrix[i]
        n_lead = int(np.argmax(~np.isnan(row))) if np.isnan(row[0]) else 0
        if n_lead:
            leading_missing[symbol] = n_lead
        if np.isnan(row[n_lead:]).any():
            interior_missing[symbol] = int(np.isnan(row[n_lead:]).sum())
    assert not interior_missing, (
        f"Round {round_number}: unexpected interior NaNs {interior_missing}"
    )
    assert leading_missing == EXPECTED_LEADING_NAN[round_number], (
        f"Round {round_number}: leading NaN pattern changed - "
        f"expected {EXPECTED_LEADING_NAN[round_number]}, found {leading_missing}"
    )

    forecast_dates = pd.bdate_range(fc_start, fc_end)
    assert len(forecast_dates) == PREDICTION_LENGTH, f"Round {round_number}: forecast window"
    return matrix, forecast_dates


def generate_round_samples(context_matrix, round_number: int):
    """Draw NUM_SAMPLES trajectories for one round; returns (100, 500, 20).

    The seed is set ONCE here and deliberately not reset between chunks:
    re-seeding would make every chunk an identical copy of the first.
    """
    torch.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed_all(RANDOM_SEED)

    chunks = []
    for chunk in range(N_CHUNKS):
        batch_outputs = []
        for start in range(0, N_ASSETS, SERIES_BATCH_SIZE):
            stop = min(start + SERIES_BATCH_SIZE, N_ASSETS)
            batch_context = [
                torch.tensor(context_matrix[i], dtype=torch.float32)
                for i in range(start, stop)
            ]
            samples = pipeline.predict(
                inputs=batch_context,
                prediction_length=PREDICTION_LENGTH,
                num_samples=SAMPLE_CHUNK_SIZE,
                temperature=TEMPERATURE,
                top_k=RESOLVED_TOP_K,
                top_p=TOP_P,
            )
            batch_outputs.append(samples.to(torch.float32).cpu().numpy())
            # Release the GPU tensor as soon as it is on the host.
            del samples, batch_context
        torch.cuda.empty_cache()

        chunk_samples = np.concatenate(batch_outputs, axis=0)
        assert chunk_samples.shape == (N_ASSETS, SAMPLE_CHUNK_SIZE, PREDICTION_LENGTH), (
            f"Round {round_number} chunk {chunk + 1}: got {chunk_samples.shape}"
        )
        chunks.append(chunk_samples)
        print(f"    chunk {chunk + 1}/{N_CHUNKS} -> {tuple(chunk_samples.shape)}")

    forecast_samples = np.concatenate(chunks, axis=1)
    assert forecast_samples.shape == (N_ASSETS, NUM_SAMPLES, PREDICTION_LENGTH), (
        f"Round {round_number}: got {forecast_samples.shape}"
    )
    # Identical chunks would mean the seed was reset mid-round.
    assert not np.array_equal(
        forecast_samples[:, :SAMPLE_CHUNK_SIZE],
        forecast_samples[:, SAMPLE_CHUNK_SIZE:2 * SAMPLE_CHUNK_SIZE]
    ), f"Round {round_number}: chunk 2 repeats chunk 1 - the seed was reset"
    return forecast_samples


def describe_values(forecast_samples) -> dict:
    """Range and finiteness of a raw forecast array, for reporting only."""
    finite = np.isfinite(forecast_samples)
    n_nonfinite = int((~finite).sum())
    values = forecast_samples[finite]
    return {
        "all_finite": n_nonfinite == 0,
        "n_nonfinite": n_nonfinite,
        "min": float(values.min()) if values.size else float("nan"),
        "max": float(values.max()) if values.size else float("nan"),
    }


def save_and_verify_round(round_number: int, forecast_samples, forecast_dates) -> Path:
    """Save the raw array plus run metadata, then reload and check it."""
    path = samples_path(round_number)
    ctx_start, origin, fc_start, fc_end = ROUND_SCHEDULE[round_number]
    date_strings = np.array([d.strftime("%Y-%m-%d") for d in forecast_dates])

    metadata = {
        "model_id": MODEL_ID,
        "run_type": "m6_holdout",
        "round": round_number,
        "origin": origin,
        "context_start": ctx_start,
        "forecast_start": fc_start,
        "forecast_end": fc_end,
        "temperature": TEMPERATURE,
        "top_k_setting": TOP_K_SETTING,
        "resolved_top_k": RESOLVED_TOP_K,
        "top_p": TOP_P,
        "num_samples": NUM_SAMPLES,
        "prediction_length": PREDICTION_LENGTH,
        "context_length": CONTEXT_LENGTH,
        "random_seed": RANDOM_SEED,
        "sample_chunk_size": SAMPLE_CHUNK_SIZE,
        "series_batch_size": SERIES_BATCH_SIZE,
        "dtype": str(DTYPE),
        "selected_on": "six pre-M6 validation origins, lowest mean RPS",
        "generated_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
        **describe_values(forecast_samples),
    }

    np.savez_compressed(
        path,
        forecast_samples=forecast_samples,
        asset_symbols=np.array(OFFICIAL_ASSET_ORDER),
        forecast_dates=date_strings,
        run_metadata=np.array(json.dumps(metadata)),
    )

    with np.load(path, allow_pickle=False) as reloaded:
        r_samples = reloaded["forecast_samples"]
        r_symbols = list(reloaded["asset_symbols"])
        r_dates = list(reloaded["forecast_dates"])
        r_meta = json.loads(str(reloaded["run_metadata"]))

    assert r_samples.shape == (N_ASSETS, NUM_SAMPLES, PREDICTION_LENGTH), "reloaded shape"
    assert r_symbols == OFFICIAL_ASSET_ORDER, "asset ordering changed on save/reload"
    assert np.array_equal(r_samples, forecast_samples, equal_nan=True), "saved values differ"
    assert r_dates == list(date_strings), "forecast dates changed on save/reload"
    assert r_meta["temperature"] == TEMPERATURE, "metadata temperature"
    assert r_meta["resolved_top_k"] == RESOLVED_TOP_K, "metadata top-k"
    return path


def already_done(round_number: int) -> bool:
    """True if this round is already saved and passes its checks.

    A file that fails any check, or was produced with different sampling
    settings, counts as not done and is regenerated.
    """
    path = samples_path(round_number)
    if not path.is_file():
        return False
    _, _, fc_start, fc_end = ROUND_SCHEDULE[round_number]
    expected_dates = [d.strftime("%Y-%m-%d") for d in pd.bdate_range(fc_start, fc_end)]
    try:
        with np.load(path, allow_pickle=False) as f:
            if not {"forecast_samples", "asset_symbols",
                    "forecast_dates", "run_metadata"} <= set(f.files):
                return False
            if f["forecast_samples"].shape != (N_ASSETS, NUM_SAMPLES, PREDICTION_LENGTH):
                return False
            if list(f["asset_symbols"]) != OFFICIAL_ASSET_ORDER:
                return False
            if list(f["forecast_dates"]) != expected_dates:
                return False
            meta = json.loads(str(f["run_metadata"]))
        return (meta.get("model_id") == MODEL_ID
                and meta.get("temperature") == TEMPERATURE
                and meta.get("resolved_top_k") == RESOLVED_TOP_K
                and meta.get("top_p") == TOP_P
                and meta.get("num_samples") == NUM_SAMPLES
                and meta.get("random_seed") == RANDOM_SEED)
    except Exception:
        # An unreadable or truncated file counts as not done.
        return False


print("Helpers defined.")

### Run all 12 M6 rounds

Each round is written and verified before the next begins. Re-running this cell
resumes: rounds already saved with these exact settings are skipped.

In [ ]:
records, failures, skipped = [], [], []
experiment_started = datetime.now(timezone.utc)

for r in ROUNDS:
    _, origin, fc_start, fc_end = ROUND_SCHEDULE[r]
    print(f"\n{'=' * 70}\nRound {r:02d} | origin {origin} | "
          f"forecast {fc_start} .. {fc_end}\n{'=' * 70}")

    if already_done(r):
        skipped.append(r)
        print("  already saved and verified, skipped")
        continue

    try:
        context_matrix, forecast_dates = load_and_validate_context(r)

        started = datetime.now(timezone.utc)
        forecast_samples = generate_round_samples(context_matrix, r)
        elapsed = (datetime.now(timezone.utc) - started).total_seconds()

        stats = describe_values(forecast_samples)
        saved_path = save_and_verify_round(r, forecast_samples, forecast_dates)
        records.append({"round": r, "origin": origin, "seconds": round(elapsed, 1),
                        "file": saved_path.name, **stats})
        flag = "" if stats["all_finite"] else f"  !! {stats['n_nonfinite']} NON-FINITE"
        print(f"  generated in {elapsed:.1f}s, range "
              f"[{stats['min']:+.4f}, {stats['max']:+.4f}]")
        print(f"  saved+verified: {saved_path.name}{flag}")

        del forecast_samples, context_matrix
        torch.cuda.empty_cache()

    except Exception as exc:
        failures.append((r, f"{type(exc).__name__}: {exc}"))
        print(f"  !! ROUND {r:02d} FAILED - completed rounds remain saved")
        traceback.print_exc()
        torch.cuda.empty_cache()

total_minutes = (datetime.now(timezone.utc) - experiment_started).total_seconds() / 60
print(f"\nFinished in {total_minutes:.1f} min. "
      f"Generated {len(records)}, skipped {len(skipped)}, failed {len(failures)} "
      f"of {len(list(ROUNDS))} rounds.")
if failures:
    print(f"FAILED rounds: {[r for r, _ in failures]}")

## 8. Verify every saved file

Re-reads all 12 files from Drive, independently of the loop above, and checks
the array shape, asset order, forecast dates and stored sampling settings.

In [ ]:
ALL_VERIFIED = True
NONFINITE_ROUNDS = []

print(f"{'rnd':>4}  {'file':<58} {'status':<8} {'min':>9} {'max':>9}")
for r in ROUNDS:
    path = samples_path(r)
    if not path.is_file():
        ALL_VERIFIED = False
        print(f"{r:>4}  {path.name:<58} MISSING")
        continue

    _, _, fc_start, fc_end = ROUND_SCHEDULE[r]
    expected_dates = [d.strftime("%Y-%m-%d") for d in pd.bdate_range(fc_start, fc_end)]
    with np.load(path, allow_pickle=False) as f:
        samples = f["forecast_samples"]
        symbols = list(f["asset_symbols"])
        fdates = list(f["forecast_dates"])
        meta = json.loads(str(f["run_metadata"]))

    stats = describe_values(samples)
    ok = (samples.shape == (N_ASSETS, NUM_SAMPLES, PREDICTION_LENGTH)
          and symbols == OFFICIAL_ASSET_ORDER
          and fdates == expected_dates
          and meta["model_id"] == MODEL_ID
          and meta["temperature"] == TEMPERATURE
          and meta["resolved_top_k"] == RESOLVED_TOP_K
          and meta["top_p"] == TOP_P
          and meta["num_samples"] == NUM_SAMPLES
          and meta["random_seed"] == RANDOM_SEED)
    ALL_VERIFIED = ALL_VERIFIED and ok
    if not stats["all_finite"]:
        NONFINITE_ROUNDS.append(r)
    print(f"{r:>4}  {path.name:<58} {'OK' if ok else 'FAILED':<8} "
          f"{stats['min']:>9.4f} {stats['max']:>9.4f}")

print(f"\nAll {len(list(ROUNDS))} files present and verified: {ALL_VERIFIED}")
if NONFINITE_ROUNDS:
    print(f"Rounds containing non-finite values, saved unchanged: {NONFINITE_ROUNDS}")
else:
    print("No non-finite values in any saved file.")

## 9. Completion summary

What was produced and where the files should go next.

In [ ]:
print("Financial Chronos 46M - final M6 holdout run")
print(f"  model      : {MODEL_ID} (loaded once, frozen, inference only)")
print(f"  sampling   : temperature {TEMPERATURE}, top_k {TOP_K_SETTING} "
      f"= {RESOLVED_TOP_K}, top_p {TOP_P}")
print(f"  fixed      : {NUM_SAMPLES} samples, context {CONTEXT_LENGTH}, "
      f"horizon {PREDICTION_LENGTH}, seed {RANDOM_SEED}, batch {SERIES_BATCH_SIZE}")
print(f"  selected on: six pre-M6 validation origins, lowest mean RPS")
print(f"  rounds     : {len(records)} generated, {len(skipped)} skipped, "
      f"{len(failures)} failed")
print(f"  verified   : {ALL_VERIFIED}")
print(f"  output     : {DRIVE_OUTPUT_DIR}")
print("  metrics    : none computed here (no RPS, ECE, confidence, entropy, shrinkage)")

saved = sum(1 for r in ROUNDS if samples_path(r).is_file())
if saved == len(list(ROUNDS)):
    print("\n  All 12 rounds COMPLETE. Copy the folder from Drive into")
    print("  TSFM Uncertainty Experiment/model_outputs/financial_chronos/m6/temp5.0_topkfull/")
    print("  then score it locally with the project's existing scoring code.")
else:
    print(f"\n  INCOMPLETE: {saved}/12 rounds saved. Re-run the loop in section 7 "
          "to resume; finished rounds are skipped.")